In [2]:
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print("Trainingsdaten:", len(train))
print("Validierungsdaten:", len(valid))
print("Testdaten:", len(test))

Trainingsdaten: 16801
Validierungsdaten: 1000
Testdaten: 1532


In [3]:

print("Spalten:", train.column_names)

print("\nErster Dateneintrag:")
print(train[0])

Spalten: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']

Erster Dateneintrag:
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


### Untersuchung der Label-Verteilung

Vor dem Training wird untersucht, wie häufig die beiden möglichen Labels im Trainingsdatensatz vorkommen.

Ziel ist es, festzustellen, ob die Klassen ungefähr gleichmässig verteilt sind oder ein Klassenungleichgewicht (*Class Imbalance*) vorliegt.

Ein starkes Ungleichgewicht könnte dazu führen, dass ein Modell bevorzugt die häufigere Klasse vorhersagt und dadurch eine hohe Accuracy erzielt, ohne die eigentliche Aufgabe zuverlässig zu lösen.

In [4]:

from collections import Counter

label_counts = Counter(train["label"])

print("Label-Verteilung:")
for label, count in sorted(label_counts.items()):
    percentage = count / len(train) * 100
    print(f"Label {label}: {count} ({percentage:.2f} %)")


Label-Verteilung:
Label 1: 8461 (50.36 %)
Label 2: 8340 (49.64 %)


In [5]:
columns = [
    "observation_1",
    "observation_2",
    "hypothesis_1",
    "hypothesis_2"
]

for column in columns:
    lengths = [len(text.split()) for text in train[column]]

    print(f"\n{column}:")
    print(f"Durchschnitt: {sum(lengths) / len(lengths):.2f} Wörter")
    print(f"Maximum: {max(lengths)} Wörter")
    print(f"Minimum: {min(lengths)} Wörter")



observation_1:
Durchschnitt: 8.08 Wörter
Maximum: 16 Wörter
Minimum: 2 Wörter

observation_2:
Durchschnitt: 9.28 Wörter
Maximum: 19 Wörter
Minimum: 2 Wörter

hypothesis_1:
Durchschnitt: 8.37 Wörter
Maximum: 30 Wörter
Minimum: 1 Wörter

hypothesis_2:
Durchschnitt: 8.39 Wörter
Maximum: 30 Wörter
Minimum: 1 Wörter


### Untersuchung der kombinierten Textlängen

Die Wortanzahl der beiden Beobachtungen und einer Hypothese wird zusammengezählt. Dadurch lässt sich abschätzen, wie lang die späteren Modelleingaben sind und welche maximale Sequenzlänge für Padding und Truncation sinnvoll ist.

In [6]:

combined_lengths = []

for example in train:
    length = (
        len(example["observation_1"].split())
        + len(example["observation_2"].split())
        + len(example["hypothesis_1"].split())
    )
    combined_lengths.append(length)

print("Durchschnitt:", round(sum(combined_lengths) / len(combined_lengths), 2))
print("Minimum:", min(combined_lengths))
print("Maximum:", max(combined_lengths))


Durchschnitt: 25.73
Minimum: 9
Maximum: 53


### Tokenisierung

Die Texte werden in einzelne Tokens zerlegt und in Kleinbuchstaben umgewandelt. Dies bildet die Grundlage für den späteren Aufbau des Vokabulars und die Umwandlung der Wörter in numerische IDs.

In [7]:

import re

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

example = train[0]["observation_1"]

print("Original:", example)
print("Tokens:", tokenize(example))


Original: A baby panda was lost in the woods.
Tokens: ['a', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods', '.']


### Erstellung des Vokabulars

Aus den Trainingsdaten wird ein Vokabular erstellt, das jedem Token eine eindeutige numerische ID zuordnet. Zusätzlich werden spezielle Tokens für Padding (`<PAD>`) und unbekannte Wörter (`<UNK>`) definiert.

Das Vokabular wird ausschliesslich anhand der Trainingsdaten aufgebaut, um Data Leakage zu vermeiden.

In [8]:

from collections import Counter

token_counts = Counter()

for example in train:
    for column in columns:
        tokens = tokenize(example[column])
        token_counts.update(tokens)

vocab = {"<PAD>": 0, "<UNK>": 1}

for token in token_counts:
    vocab[token] = len(vocab)

print("Vokabulargrösse:", len(vocab))
print("Erste 15 Einträge:", list(vocab.items())[:15])

Vokabulargrösse: 15818
Erste 15 Einträge: [('<PAD>', 0), ('<UNK>', 1), ('a', 2), ('baby', 3), ('panda', 4), ('was', 5), ('lost', 6), ('in', 7), ('the', 8), ('woods', 9), ('.', 10), ('he', 11), ('reunited', 12), ('with', 13), ('his', 14)]


### Untersuchung seltener Tokens

Es wird untersucht, wie viele Tokens nur selten im Trainingsdatensatz vorkommen. Ziel ist es, zu entscheiden, ob eine Mindesthäufigkeit für die Aufnahme ins Vokabular sinnvoll ist, um die Anzahl selten trainierter Word Embeddings zu reduzieren.

In [9]:

print("Tokens mit Häufigkeit 1:",
      sum(count == 1 for count in token_counts.values()))

print("Tokens mit Häufigkeit unter 3:",
      sum(count < 3 for count in token_counts.values()))

print("Tokens mit Häufigkeit unter 5:",
      sum(count < 5 for count in token_counts.values()))

print("\nDie 10 häufigsten Tokens:")
print(token_counts.most_common(10))


Tokens mit Häufigkeit 1: 4832
Tokens mit Häufigkeit unter 3: 7177
Tokens mit Häufigkeit unter 5: 9629

Die 10 häufigsten Tokens:
[('.', 65608), ('the', 27649), ('to', 23083), ('a', 20897), ('was', 15793), ('and', 13201), ('her', 9465), ("'", 8435), ('he', 8383), ('his', 8122)]


### Begrenzung des Vokabulars

Für das Vokabular wird eine Mindesthäufigkeit von drei Vorkommen festgelegt. Seltenere Tokens werden durch `<UNK>` repräsentiert. Dadurch wird die Anzahl der zu trainierenden Word Embeddings reduziert.

In [10]:

min_freq = 3

vocab = {"<PAD>": 0, "<UNK>": 1}

for token, count in token_counts.items():
    if count >= min_freq:
        vocab[token] = len(vocab)

print("Neue Vokabulargrösse:", len(vocab))
print("Entfernte Tokens:", len(token_counts) + 2 - len(vocab))


Neue Vokabulargrösse: 8641
Entfernte Tokens: 7177


### Umwandlung von Tokens in numerische IDs

Die tokenisierten Texte werden mithilfe des erstellten Vokabulars in numerische IDs umgewandelt. Tokens, die nicht im Vokabular enthalten sind, erhalten die ID des speziellen Tokens `<UNK>`.

In [11]:

def encode(text, vocab):
    tokens = tokenize(text)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

example = train[0]["observation_1"]

print("Original:", example)
print("Tokens:", tokenize(example))
print("Token-IDs:", encode(example, vocab))


Original: A baby panda was lost in the woods.
Tokens: ['a', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods', '.']
Token-IDs: [2, 3, 4, 5, 6, 7, 8, 9, 10]


### Umwandlung in PyTorch-Tensoren

Die numerischen Token-IDs werden in PyTorch-Tensoren umgewandelt. Diese bilden die Grundlage für die spätere Verarbeitung durch die Embedding-Schicht und das neuronale Netzwerk.

In [12]:

import torch

token_ids = encode(example, vocab)

tensor = torch.tensor(token_ids, dtype=torch.long)

print("Tensor:", tensor)
print("Datentyp:", tensor.dtype)
print("Form:", tensor.shape)


Tensor: tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10])
Datentyp: torch.int64
Form: torch.Size([9])


### Erstellung von Word Embeddings

Mithilfe einer PyTorch-Embedding-Schicht werden die numerischen Token-IDs in Vektoren umgewandelt. Für jedes Token wird ein Vektor mit 50 Dimensionen erstellt. Die Embeddings werden zunächst zufällig initialisiert und später während des Modelltrainings angepasst.

In [13]:

embedding_dim = 50

embedding = torch.nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=embedding_dim,
    padding_idx=vocab["<PAD>"]
)

embedded = embedding(tensor)

print("Ursprüngliche Form:", tensor.shape)
print("Embedding-Form:", embedded.shape)
print("\nErster Embedding-Vektor:")
print(embedded[0])


Ursprüngliche Form: torch.Size([9])
Embedding-Form: torch.Size([9, 50])

Erster Embedding-Vektor:
tensor([ 1.2708e+00, -5.4127e-01,  5.7411e-01, -5.7733e-01, -2.0320e+00,
        -6.1655e-03, -1.6260e-01,  6.6656e-01,  1.1455e+00,  8.1248e-01,
         1.1682e-01,  8.1455e-01,  8.3071e-01,  6.0938e-01, -8.4802e-04,
         4.8574e-01,  1.7525e+00, -1.3207e+00,  5.6436e-01,  1.1370e+00,
         1.7446e+00,  8.4605e-01,  5.1332e-01, -1.1551e+00,  6.5441e-01,
        -1.3458e+00,  2.9829e-02, -4.5277e-01, -1.3693e+00,  1.4892e+00,
         7.6967e-01, -2.2387e-01,  3.3669e-01, -1.1902e+00,  2.0842e-01,
        -3.9619e-01,  1.7807e-01, -5.5054e-01, -1.2615e-01,  8.6251e-01,
        -8.3247e-01, -1.9957e+00, -1.8487e-01,  9.3630e-01,  1.1217e+00,
         1.4494e+00,  6.9495e-01, -4.6260e-01,  6.8531e-03,  5.2306e-01],
       grad_fn=<SelectBackward0>)


### 2.13 Mean Pooling – Satzrepräsentation

Die Embedding-Schicht erzeugt für jedes Token einen Vektor mit 50 Dimensionen. Mithilfe von Mean Pooling wird über alle Token-Vektoren der Durchschnitt berechnet. Dadurch entsteht ein einzelner Vektor mit 50 Dimensionen, der den gesamten Satz repräsentiert.

Bei dieser Methode geht die Reihenfolge der Wörter verloren.

In [14]:
sentence_embedding = embedded.mean(dim=0)

print("Vor Mean Pooling:", embedded.shape)
print("Nach Mean Pooling:", sentence_embedding.shape)

Vor Mean Pooling: torch.Size([9, 50])
Nach Mean Pooling: torch.Size([50])


### 2.14 Kodierung der Beobachtungen und Hypothesen

Für jedes Beispiel werden beide Beobachtungen und beide Hypothesen tokenisiert und mithilfe des zuvor erstellten Vokabulars in numerische Token-IDs umgewandelt. Die Länge der resultierenden Sequenzen hängt vom jeweiligen Text ab.

Diese Sequenzen bilden die Grundlage für die spätere Bewertung der beiden Hypothesen.

In [15]:
example = train[0]

for column in columns:
    token_ids = encode(example[column], vocab)

    print(f"{column}:")
    print(f"  Text: {example[column]}")
    print(f"  Token-IDs: {token_ids}")
    print(f"  Anzahl Tokens: {len(token_ids)}")
    print()

observation_1:
  Text: A baby panda was lost in the woods.
  Token-IDs: [2, 3, 4, 5, 6, 7, 8, 9, 10]
  Anzahl Tokens: 9

observation_2:
  Text: He was reunited with his parents, who also lived at the zoo.
  Token-IDs: [11, 5, 12, 13, 14, 15, 16, 17, 18, 19, 20, 8, 21, 10]
  Anzahl Tokens: 14

hypothesis_1:
  Text: He was never lost.
  Token-IDs: [11, 5, 22, 6, 10]
  Anzahl Tokens: 5

hypothesis_2:
  Text: All the people from the zoo looked for him.
  Token-IDs: [23, 8, 24, 25, 8, 21, 26, 27, 28, 10]
  Anzahl Tokens: 10



### 2.15 Padding von Token-Sequenzen

Die vier Texte eines Trainingsbeispiels enthalten unterschiedlich viele Tokens. Mithilfe von Padding werden kürzere Sequenzen mit der speziellen Token-ID `0` (`<PAD>`) ergänzt, sodass alle Sequenzen dieselbe Länge besitzen.

Die zusätzlichen Padding-Tokens enthalten keine sprachliche Information und müssen bei der späteren Verarbeitung entsprechend berücksichtigt werden.

In [16]:
from torch.nn.utils.rnn import pad_sequence

sequences = [
    torch.tensor(encode(example[column], vocab), dtype=torch.long)
    for column in columns
]

padded = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=vocab["<PAD>"]
)

print("Form nach Padding:", padded.shape)
print("\nHypothesis 1 nach Padding:")
print(padded[2])

Form nach Padding: torch.Size([4, 14])

Hypothesis 1 nach Padding:
tensor([11,  5, 22,  6, 10,  0,  0,  0,  0,  0,  0,  0,  0,  0])


### 2.16 Padding-Maske erstellen

Durch Padding enthalten die Sequenzen zusätzliche Platzhalter (`<PAD>`), damit alle Texte dieselbe Länge haben.
Diese Platzhalter gehören nicht zum ursprünglichen Text und sollen die Berechnung der Satzrepräsentation nicht beeinflussen.
Mit einer Padding-Maske wird festgehalten, welche Positionen echte Tokens enthalten und welche lediglich Padding sind.

In [17]:
padding_mask = padded != vocab["<PAD>"]

print("Hypothesis 1:")
print(padded[2])

print("\nPadding-Maske:")
print(padding_mask[2])

Hypothesis 1:
tensor([11,  5, 22,  6, 10,  0,  0,  0,  0,  0,  0,  0,  0,  0])

Padding-Maske:
tensor([ True,  True,  True,  True,  True, False, False, False, False, False,
        False, False, False, False])


### 2.17 Mean Pooling mit Padding-Maske

Beim normalen Mean Pooling wird der Durchschnitt über alle Token-Embeddings berechnet. Nach dem Padding würde dies auch die künstlich hinzugefügten Platzhalter berücksichtigen.

Mithilfe der zuvor erstellten Padding-Maske werden diese Platzhalter ausgeschlossen. Der Durchschnitt wird dadurch nur über die tatsächlichen Tokens des Textes berechnet.

Das Ergebnis ist für jeden Text ein einzelner Vektor mit 50 Dimensionen, unabhängig von seiner ursprünglichen Länge.

In [18]:
# Embeddings für alle vier Texte erstellen
embeddings = embedding(padded)

# Padding-Maske für die Embeddings vorbereiten
mask = padding_mask.unsqueeze(-1).float()

# Padding-Positionen auf 0 setzen
masked_embeddings = embeddings * mask

# Summe der echten Token-Embeddings berechnen
summed_embeddings = masked_embeddings.sum(dim=1)

# Anzahl echter Tokens pro Text bestimmen
token_counts = mask.sum(dim=1).clamp(min=1)

# Durchschnitt berechnen
sentence_embeddings = summed_embeddings / token_counts

print("Vor Mean Pooling:", embeddings.shape)
print("Nach Mean Pooling:", sentence_embeddings.shape)

Vor Mean Pooling: torch.Size([4, 14, 50])
Nach Mean Pooling: torch.Size([4, 50])


### 2.18 Kombination der Beobachtungen und Hypothesen

Für jeden der vier Texte wurde durch Masked Mean Pooling ein Vektor mit 50 Dimensionen erstellt.

Um die Plausibilität der beiden Hypothesen später bewerten zu können, werden die Vektoren der beiden Beobachtungen jeweils mit dem Vektor einer Hypothese verbunden.

Durch diese sogenannte Concatenation entstehen zwei Vektoren mit jeweils 150 Dimensionen. Diese können anschliessend von einem neuronalen Netz verarbeitet werden.

In [19]:
# Einzelne Satzvektoren auswählen
obs1 = sentence_embeddings[0]
obs2 = sentence_embeddings[1]
hyp1 = sentence_embeddings[2]
hyp2 = sentence_embeddings[3]

# Beobachtungen mit den jeweiligen Hypothesen kombinieren
combination_1 = torch.cat([obs1, obs2, hyp1], dim=0)
combination_2 = torch.cat([obs1, obs2, hyp2], dim=0)

print("Observation 1:", obs1.shape)
print("Observation 2:", obs2.shape)
print("Hypothesis 1:", hyp1.shape)
print("Hypothesis 2:", hyp2.shape)

print("\nKombination 1:", combination_1.shape)
print("Kombination 2:", combination_2.shape)

Observation 1: torch.Size([50])
Observation 2: torch.Size([50])
Hypothesis 1: torch.Size([50])
Hypothesis 2: torch.Size([50])

Kombination 1: torch.Size([150])
Kombination 2: torch.Size([150])
